# Transcriber

## Зависимости

In [236]:
%pip install pandas

import pandas as pd
import io

from pandas import DataFrame
from typing import Set, Dict

from dataclasses import dataclass, replace
from typing import Callable, List

Note: you may need to restart the kernel to use updated packages.


## Константы

In [237]:
VOWELS: Set[str] = set('аеёиоуыэюя')
VOWEL_PHONEMES: Set[str] = set('aeoiuy')

FORBIDDEN_CHARS: Set[str] = set('ъ')

LETTER_TRANS_MAPPING: Dict[str, str] = {
    # Гласные
    'а': 'a', 'е': 'e', 'ё': 'o', 'и': 'i', 'о': 'o', 
    'у': 'u', 'ы': 'y', 'э': 'e', 'ю': 'u', 'я': 'a',

    # Согласные
    "б": "b",  "п": "p",  "в": "v",  "ф": "f",  "г": "g",  "к": "k",
    "д": "d",  "т": "t",  "з": "z",  "с": "s",  "ж": "zh", "ш": "sh",
    "х": "h",  "ц": "c",  "ч": "ch", "щ": "sc",
    "л": "l",  "м": "m",  "н": "n",  "р": "r",  "й": "j",
}

# Сопоставление звуков
VOICELESS_PAIR: Dict[str, str] = {
    "b": "p",
    "d": "t",
    "g": "k",
    "v": "f",
    "z": "s",
    "zh": "sh",
}

NEVER_MARK_PALATAL: Set[str] = {"zh", "sh", "c", "ch", "sc"}
ALWAYS_HARD: Set[str] = {"zh", "sh", "c"}
INHERENTLY_SOFT: Set[str] = {"ch", "sc", "j"}

OBSTRUENTS: Set[str] = {
    "b", "p", "d", "t", "g", "k", "v", "f",
    "z", "s", "zh", "sh", "c", "ch", "sc", "h",
}

# Пары по звонкости
VOICED_TO_VOICELESS: Dict[str, str] = {
    "b": "p", "d": "t", "g": "k",
    "v": "f", "z": "s", "zh": "sh",
}
VOICELESS_TO_VOICED: Dict[str, str] = {v: k for k, v in VOICED_TO_VOICELESS.items()}

VOICED_OBSTRUENTS: Set[str] = set(VOICED_TO_VOICELESS.keys())
VOICELESS_OBSTRUENTS: Set[str] = set(VOICED_TO_VOICELESS.values()) | {"h", "c", "ch", "sc"}

# Колонки для анализа
COL_WORD: str = 'Слово'
COL_ORTHOEPIC_TRANS: str = 'Орфоэпическая транскрипция'
COL_STRESSED_WORD: str = 'Слово с ударением'
COL_MORPHEMES = 'Слово с делением на морфемы'

# Базовые классы и функции для переиспользования

## Тестовые данные

In [238]:
# Набор тестовых слов для отладки правил
TEST_WORDS = [
    "моя0", "пое0хать", "бежи0т", "бе0лого",
    "краси0вая", "си0няя", "де0тский", "жи0ть", "щу0ка",

    # шаг 4 — ассимиляция по мягкости
    "пе0сня",      # с перед н' → s' n'
    "ба0нтик",     # н перед т' → n' t'
    "ле0стница",   # с перед т (твёрдым в кластере)

    # шаг 5 — аканье
    "молоко0",     # о о о0 → a a o0
    "хорошо0",     # о о о0 → a a o0

    # шаг 7 — ыканье
    "жена0",       # безударное е после ж → y
    "ше0сть",      # ударное е после ш — остаётся
    "цена0",       # безударное е после ц → y

    # шаг 8 — ассимиляция по звонкости
    "тру0бка",     # б перед к → p
    "сде0лать",    # с перед д → z

    # шаг 9 — конечное оглушение
    "гри0б",       # б на конце → p
    "моро0з",      # з на конце → s
    "но0ж",        # ж на конце → sh

    # шаг 10 — г → в
    "я0ркого",     # -ого → -ава

    # шаг 11 — слияние кластеров
    "сши0ть",      # сш → ш
    "мо0ются",     # тс → ц (через -ться)

    # шаг 12 — непроизносимые согласные
    "че0стный",    # стн → сн
    "по0здно",     # здн → зн
    "со0лнце",     # лнц → нц (но этого нет в твоей таблице — посмотрим)
]

## Класс фонем и функции хелперы

In [239]:
@dataclass
class Phoneme:
    base: str
    palatalized: bool = False
    stressed: bool = False
    src_pos: int = -1

    def __str__(self) -> str:
        suffix = ""
        if self.palatalized:
            suffix += "'"
        if self.stressed:
            suffix += "0"
        return self.base + suffix

def is_vowel(p: Phoneme) -> bool:
    return p.base in VOWEL_PHONEMES

def is_consonant(p: Phoneme) -> bool:
    return p.base not in VOWEL_PHONEMES

def phonemes_to_str(phonemes: List[Phoneme]) -> str:
    return " ".join(str(p) for p in phonemes)

# Вывод фонемы
def show_step(step_name: str, snapshots: Dict[str, List[Phoneme]]) -> None:
    print(f"_____________ {step_name} _____________")
    for word in TEST_WORDS:
        print(f"  {word:<14} → {phonemes_to_str(snapshots[word])}")
    print()

def parse_morpheme_boundaries(morph_str: str) -> Set[int]:
    boundaries: Set[int] = set()
    letter_idx = 0
    for ch in morph_str:
        if ch in ('*', '+', '/'):
            boundaries.add(letter_idx)
        else:
            letter_idx += 1
    return boundaries

# Чтение словаря

Файл должен находится в той же директории, что и скрипт в папке `data` с названием `my_dictionary.csv`. После загрузки данные из файла будут доступны в переменной `df`.

In [240]:
def printDataFrameInfo(data_frame):
    print(f'''
Всего слов в наборе: {len(data_frame)}

Колонки словаря:
{data_frame.columns.tolist()}

Первые 10 слов:
{data_frame.head(10)}
    ''')

df = pd.read_csv('./my_dictionary.csv', sep=',')

printDataFrameInfo(df)



Всего слов в наборе: 7303

Колонки словаря:
['Слово', 'Слово с делением на морфемы', 'Орфоэпическая транскрипция', 'Реальное произнесение', 'Количество словоупотреблений', 'Файлы']

Первые 10 слов:
           Слово Слово с делением на морфемы  \
0      абсолютно                *абсолют+н+о   
1        абхазец                   *абхаз+ец   
2         август                     *август   
3        августе                   *август/е   
4  автобиографии           *авто*био*графи/и   
5       автобусе                  *автобус/е   
6     автобусной               *автобус+н/ой   
7      автобусом                 *автобус/ом   
8        автомат                    *автомат   
9       автомата                  *автомат/а   

                   Орфоэпическая транскрипция  \
0                * a1 p s a1 l' u0 t + n + a4   
1                       * a1 p h a0 z' + i4 c   
2                             * a0 v g u4 s t   
3                      * a0 v g u4 s' t' / i4   
4  * a1 f t a2 * b' i1 a1 *

## Нормализация словаря и валидация слов

Удаляем пустые слова, дубликаты слов. Убираем лишние колонки. Сразу проверяем невалидные символы в слове.

In [241]:
df = (
    df[[COL_WORD, COL_ORTHOEPIC_TRANS, COL_MORPHEMES]]
      .dropna(subset=[COL_WORD, COL_ORTHOEPIC_TRANS])
      .drop_duplicates(subset=[COL_WORD], keep='first')
      .loc[lambda d: ~d[COL_WORD].map(
          lambda s: any(ch in FORBIDDEN_CHARS for ch in s)
      )]
      .reset_index(drop=True)
)

printDataFrameInfo(df)


Всего слов в наборе: 7226

Колонки словаря:
['Слово', 'Орфоэпическая транскрипция', 'Слово с делением на морфемы']

Первые 10 слов:
           Слово                  Орфоэпическая транскрипция  \
0      абсолютно                * a1 p s a1 l' u0 t + n + a4   
1        абхазец                       * a1 p h a0 z' + i4 c   
2         август                             * a0 v g u4 s t   
3        августе                      * a0 v g u4 s' t' / i4   
4  автобиографии  * a1 f t a2 * b' i1 a1 * g r a0 f' i4 / i4   
5       автобусе                    * a1 f t o0 b u4 s' / i4   
6     автобусной               * a1 f t o0 b u4 s + n / a4 j   
7      автобусом                   * a1 f t o0 b u4 s / a4 m   
8        автомат                          * a1 f t a1 m a0 t   
9       автомата                     * a1 f t a1 m a0 t / a4   

  Слово с делением на морфемы  
0                *абсолют+н+о  
1                   *абхаз+ец  
2                     *август  
3                   *август/е  
4 

## Расстановка ударений

Определяет, которая по счёту гласная является ударной
в транскрипции словаря.

Логика: ударная гласная помечена цифрой 0 (u0, a0, o0...).
Считаем только гласные токены (вида буква+цифра),
возвращаем порядковый номер ударной.

Пример:

> '* a1 p s a1 l' u0 t + n + a4'
>
> гласные по порядку: a1(1), a1(2), u0(3*), a4(4)
>
> возвращает 3


In [242]:
def find_stress_vowel_in_transcription(transcription: str) -> int | None:
    tokens: list[str] = transcription.lower().split(' ')

    vowel_count: int = 0
    for token in tokens:
        if(len(token) > 1 and token[0] in VOWEL_PHONEMES and token[-1].isdigit()):
            vowel_count += 1
            if token[-1] == '0':
                return vowel_count

    return None

def insert_stress(word: str, stress_vowel_pos: int) -> str:
    vowels_count: int = 0
    res: list = []
    for letter in word.lower():
        res.append(letter)
        if(letter in VOWELS):
            vowels_count += 1
            if(vowels_count == stress_vowel_pos):
                res.append('0')

    return ''.join(res)

def restore_stress_by_transcription(raw_word: str, transcription: str) -> str | None:
    stress_pos: int = find_stress_vowel_in_transcription(transcription)

    if(stress_pos is None):
        return None

    return insert_stress(raw_word, stress_pos)

df[COL_STRESSED_WORD] = [
    restore_stress_by_transcription(str(word), str(trans))
    for word, trans in zip(
        df[COL_WORD],
        df[COL_ORTHOEPIC_TRANS]
    )
]

def is_valid(word: str) -> bool:
    idx = word.find("0")
    if idx != -1:
        if idx - 1 < 0 or word[idx - 1] not in VOWELS:
            return False
    return True

mask_invalid_stress = ~df[COL_STRESSED_WORD].map(is_valid, na_action='ignore').fillna(False).astype(bool)
print(mask_invalid_stress.sum())
failed = df[ df[COL_STRESSED_WORD].isna() | mask_invalid_stress ]
print('Невалидные слова:')
printDataFrameInfo(failed)
print('-'*70)

df = df.dropna(subset=[COL_STRESSED_WORD]).reset_index(drop=True)

printDataFrameInfo(df)

8
Невалидные слова:

Всего слов в наборе: 8

Колонки словаря:
['Слово', 'Орфоэпическая транскрипция', 'Слово с делением на морфемы', 'Слово с ударением']

Первые 10 слов:
              Слово               Орфоэпическая транскрипция  \
40               ан                                   * a1 n   
98              без                                * b' i1 s   
2109  кабетришников  * k a2 b i1 t r' i1 sh + n' i1 k + a2 f   
3467             об                                   * a1 b   
6309           тебя                           * t' i1 b' / a   
6458       тридцати                 * t r' i1 * c a1 t' / i4   
7193           юго-                            * j u1 g + a1   
7210            яма                            * j a4 m / a4   

     Слово с делением на морфемы Слово с ударением  
40                           *ан               NaN  
98                          *без               NaN  
2109            *кабетриш+ник/ов               NaN  
3467                         *об        

# 1. Йотация, палатализация, Г -> В в окончании и базовый маппинг

Правила 1, 3, 10 и базовый маппинг букв в звуки. Палатализация вшита внутрь функции перевода букв в фонемы.

In [243]:
IOTATED_LETTERS: Set[str] = set("еёюя")
SOFTENING_LETTERS: Set[str] = set("еёюяи")

def rule_1_iotation(stressed_word: str) -> tuple[str, List[int]]:
    result_chars: List[str] = []
    pos_map: List[int] = []

    letter_idx = -1  # индекс ПОСЛЕДНЕЙ обработанной буквы

    for i, ch in enumerate(stressed_word):
        if ch == '0':
            # ударение приписывается к предыдущей букве
            result_chars.append(ch)
            pos_map.append(letter_idx)
            continue

        # ch — это буква (или ь). Это новая буква в исходном слове.
        new_letter_idx = letter_idx + 1

        # йотация: вставка j ПЕРЕД ch
        if ch in IOTATED_LETTERS:
            prev = None
            for k in range(i - 1, -1, -1):
                if stressed_word[k] != '0':
                    prev = stressed_word[k]
                    break
            if prev is None or prev in VOWELS or prev in ('ь', 'ъ'):
                result_chars.append('j')
                pos_map.append(new_letter_idx)  # j относится к будущей букве

        result_chars.append(ch)
        pos_map.append(new_letter_idx)
        letter_idx = new_letter_idx

    return ''.join(result_chars), pos_map

def letters_to_phonemes(stressed_word: str, pos_map: List[int]) -> List[Phoneme]:
    phonemes: List[Phoneme] = []

    for i, ch in enumerate(stressed_word):
        if ch == '0':
            if phonemes and is_vowel(phonemes[-1]):
                phonemes[-1] = replace(phonemes[-1], stressed=True)
            continue
        if ch == 'ь':
            if (phonemes and is_consonant(phonemes[-1])
                    and phonemes[-1].base not in NEVER_MARK_PALATAL):
                phonemes[-1] = replace(phonemes[-1], palatalized=True)
            continue
        if ch == 'j':
            phonemes.append(Phoneme(base='j', src_pos=pos_map[i]))
            continue
        if ch in LETTER_TRANS_MAPPING:
            if (ch in SOFTENING_LETTERS
                    and phonemes
                    and is_consonant(phonemes[-1])
                    and phonemes[-1].base not in NEVER_MARK_PALATAL
                    and phonemes[-1].base != 'j'):
                phonemes[-1] = replace(phonemes[-1], palatalized=True)
            phonemes.append(Phoneme(
                base=LETTER_TRANS_MAPPING[ch],
                src_pos=pos_map[i],
            ))

    return phonemes

def rule_10_adjective_genitive(
    stressed_word: str,
    morph_boundaries: Set[int] | None = None,
) -> str:
    clean = stressed_word.replace('0', '')
    if len(clean) < 3:
        return stressed_word
    if not (clean[-1] == 'о' and clean[-2] == 'г' and clean[-3] in ('о', 'е', 'ё')):
        return stressed_word

    ending_start_idx = len(clean) - 3

    if morph_boundaries is not None and ending_start_idx not in morph_boundaries:
        return stressed_word

    last_o_idx = stressed_word.rfind('о')
    g_idx = stressed_word.rfind('г', 0, last_o_idx)
    if g_idx == -1:
        return stressed_word
    return stressed_word[:g_idx] + 'в' + stressed_word[g_idx + 1:]

def to_phonemes(stressed_word: str, morph_boundaries: Set[int] | None = None, use_rule_10: bool = True,) -> List[Phoneme]:
    w = rule_10_adjective_genitive(stressed_word, morph_boundaries) if use_rule_10 else stressed_word
    w, pos_map = rule_1_iotation(w)
    return letters_to_phonemes(w, pos_map)

state = {w: to_phonemes(w, use_rule_10=True) for w in TEST_WORDS}
show_step("1. Йотация и маппинг", state)

_____________ 1. Йотация и маппинг _____________
  моя0           → m o j a0
  пое0хать       → p o j e0 h a t'
  бежи0т         → b' e zh i0 t
  бе0лого        → b' e0 l o v o
  краси0вая      → k r a s' i0 v a j a
  си0няя         → s' i0 n' a j a
  де0тский       → d' e0 t s k' i j
  жи0ть          → zh i0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s n' a
  ба0нтик        → b a0 n t' i k
  ле0стница      → l' e0 s t n' i c a
  молоко0        → m o l o k o0
  хорошо0        → h o r o sh o0
  жена0          → zh e n a0
  ше0сть         → sh e0 s t'
  цена0          → c e n a0
  тру0бка        → t r u0 b k a
  сде0лать       → s d' e0 l a t'
  гри0б          → g r' i0 b
  моро0з         → m o r o0 z
  но0ж           → n o0 zh
  я0ркого        → j a0 r k o v o
  сши0ть         → s sh i0 t'
  мо0ются        → m o0 j u t s' a
  че0стный       → ch e0 s t n y j
  по0здно        → p o0 z d n o
  со0лнце        → s o0 l n c e



# 2. Выпадение интервокального j

In [244]:
def rule_2_intervocalic_jot(phonemes: List[Phoneme]) -> List[Phoneme]:
    PRESERVED_BEFORE = {"u", "o"}
    
    result: List[Phoneme] = []
    i = 0
    while i < len(phonemes):
        p = phonemes[i]
        if (p.base == 'j'
                and result and is_vowel(result[-1])
                and i + 1 < len(phonemes)
                and is_vowel(phonemes[i + 1])
                and not phonemes[i + 1].stressed
                and phonemes[i + 1].base not in PRESERVED_BEFORE):
            next_vowel = replace(phonemes[i + 1], base='i')
            result.append(next_vowel)
            i += 2
            continue
        result.append(p)
        i += 1
    return result


state = {w: rule_2_intervocalic_jot(state[w]) for w in TEST_WORDS}
show_step("2. Выпадение интервокального j", state)

_____________ 2. Выпадение интервокального j _____________
  моя0           → m o j a0
  пое0хать       → p o j e0 h a t'
  бежи0т         → b' e zh i0 t
  бе0лого        → b' e0 l o v o
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' a i
  де0тский       → d' e0 t s k' i j
  жи0ть          → zh i0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s n' a
  ба0нтик        → b a0 n t' i k
  ле0стница      → l' e0 s t n' i c a
  молоко0        → m o l o k o0
  хорошо0        → h o r o sh o0
  жена0          → zh e n a0
  ше0сть         → sh e0 s t'
  цена0          → c e n a0
  тру0бка        → t r u0 b k a
  сде0лать       → s d' e0 l a t'
  гри0б          → g r' i0 b
  моро0з         → m o r o0 z
  но0ж           → n o0 zh
  я0ркого        → j a0 r k o v o
  сши0ть         → s sh i0 t'
  мо0ются        → m o0 j u t s' a
  че0стный       → ch e0 s t n y j
  по0здно        → p o0 z d n o
  со0лнце        → s o0 l n c e



# 3. Регрессивная ассимиляция по мягкости

In [245]:
SOFTNESS_ASSIMILATION_PAIRS: Set[tuple] = {
    ("s", "t"), ("z", "t"), ("s", "d"), ("z", "d"),
    ("s", "n"), ("z", "n"),
    ("n", "s"), ("n", "z"), ("n", "t"), ("n", "d"),
    ("n", "ch"), ("n", "sc"),
    ("t", "n"), ("d", "n"),
    ("g", "k"),
}

def rule_4_softness_assimilation(phonemes: List[Phoneme]) -> List[Phoneme]:
    result = phonemes[:]
    for i in range(len(result) - 2, -1, -1):
        left, right = result[i], result[i + 1]
        right_is_soft = right.palatalized or right.base in INHERENTLY_SOFT
        if (is_consonant(left) and is_consonant(right)
                and right_is_soft
                and not left.palatalized
                and left.base not in NEVER_MARK_PALATAL
                and (left.base, right.base) in SOFTNESS_ASSIMILATION_PAIRS):
            result[i] = replace(left, palatalized=True)
    return result

state = {w: rule_4_softness_assimilation(state[w]) for w in TEST_WORDS}
show_step("3. Ассимиляция по мягкости", state)

_____________ 3. Ассимиляция по мягкости _____________
  моя0           → m o j a0
  пое0хать       → p o j e0 h a t'
  бежи0т         → b' e zh i0 t
  бе0лого        → b' e0 l o v o
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' a i
  де0тский       → d' e0 t s k' i j
  жи0ть          → zh i0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s' n' a
  ба0нтик        → b a0 n' t' i k
  ле0стница      → l' e0 s' t' n' i c a
  молоко0        → m o l o k o0
  хорошо0        → h o r o sh o0
  жена0          → zh e n a0
  ше0сть         → sh e0 s' t'
  цена0          → c e n a0
  тру0бка        → t r u0 b k a
  сде0лать       → s' d' e0 l a t'
  гри0б          → g r' i0 b
  моро0з         → m o r o0 z
  но0ж           → n o0 zh
  я0ркого        → j a0 r k o v o
  сши0ть         → s sh i0 t'
  мо0ются        → m o0 j u t s' a
  че0стный       → ch e0 s t n y j
  по0здно        → p o0 z d n o
  со0лнце        → s o0 l n c e



# 4. Аканье

In [246]:
def rule_5_akanye(phonemes: List[Phoneme]) -> List[Phoneme]:
    return [
        replace(p, base='a') if p.base == 'o' and not p.stressed else p
        for p in phonemes
    ]


state = {w: rule_5_akanye(state[w]) for w in TEST_WORDS}
show_step("5. Аканье", state)

_____________ 5. Аканье _____________
  моя0           → m a j a0
  пое0хать       → p a j e0 h a t'
  бежи0т         → b' e zh i0 t
  бе0лого        → b' e0 l a v a
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' a i
  де0тский       → d' e0 t s k' i j
  жи0ть          → zh i0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s' n' a
  ба0нтик        → b a0 n' t' i k
  ле0стница      → l' e0 s' t' n' i c a
  молоко0        → m a l a k o0
  хорошо0        → h a r a sh o0
  жена0          → zh e n a0
  ше0сть         → sh e0 s' t'
  цена0          → c e n a0
  тру0бка        → t r u0 b k a
  сде0лать       → s' d' e0 l a t'
  гри0б          → g r' i0 b
  моро0з         → m a r o0 z
  но0ж           → n o0 zh
  я0ркого        → j a0 r k a v a
  сши0ть         → s sh i0 t'
  мо0ются        → m o0 j u t s' a
  че0стный       → ch e0 s t n y j
  по0здно        → p o0 z d n a
  со0лнце        → s o0 l n c e



# 5. Иканье

In [247]:
def rule_6_ikanye(phonemes: List[Phoneme]) -> List[Phoneme]:
    last_vowel_idx = -1
    for i in range(len(phonemes) - 1, -1, -1):
        if is_vowel(phonemes[i]):
            last_vowel_idx = i
            break

    result: List[Phoneme] = []
    for i, p in enumerate(phonemes):
        if p.stressed or p.base not in ('e', 'a'):
            result.append(p)
            continue

        prev = phonemes[i - 1] if i > 0 else None

        if (i == last_vowel_idx
                and p.base == 'a'
                and prev is not None
                and is_consonant(prev)
                and (prev.palatalized or prev.base in INHERENTLY_SOFT)):
            result.append(p)
            continue

        after_soft = prev is not None and (
            (is_consonant(prev) and prev.palatalized)
            or prev.base in INHERENTLY_SOFT
        )
        at_word_start = prev is None

        if after_soft or (p.base == 'e' and at_word_start):
            result.append(replace(p, base='i'))
        else:
            result.append(p)
    return result

state = {w: rule_6_ikanye(state[w]) for w in TEST_WORDS}
show_step("6. Иканье", state)

_____________ 6. Иканье _____________
  моя0           → m a j a0
  пое0хать       → p a j e0 h a t'
  бежи0т         → b' i zh i0 t
  бе0лого        → b' e0 l a v a
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' i i
  де0тский       → d' e0 t s k' i j
  жи0ть          → zh i0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s' n' a
  ба0нтик        → b a0 n' t' i k
  ле0стница      → l' e0 s' t' n' i c a
  молоко0        → m a l a k o0
  хорошо0        → h a r a sh o0
  жена0          → zh e n a0
  ше0сть         → sh e0 s' t'
  цена0          → c e n a0
  тру0бка        → t r u0 b k a
  сде0лать       → s' d' e0 l a t'
  гри0б          → g r' i0 b
  моро0з         → m a r o0 z
  но0ж           → n o0 zh
  я0ркого        → j a0 r k a v a
  сши0ть         → s sh i0 t'
  мо0ются        → m o0 j u t s' a
  че0стный       → ch e0 s t n y j
  по0здно        → p o0 z d n a
  со0лнце        → s o0 l n c e



# 6. Ыканье

In [248]:
def rule_7_ykanye(phonemes: List[Phoneme]) -> List[Phoneme]:
    result: List[Phoneme] = []
    for i, p in enumerate(phonemes):
        prev = phonemes[i - 1] if i > 0 else None
        after_hard_hush = prev is not None and prev.base in ALWAYS_HARD

        if after_hard_hush and p.base == 'i':
            result.append(replace(p, base='y'))
        elif after_hard_hush and p.base == 'e' and not p.stressed:
            result.append(replace(p, base='y'))
        else:
            result.append(p)
    return result


state = {w: rule_7_ykanye(state[w]) for w in TEST_WORDS}
show_step("7. Ыканье", state)

_____________ 7. Ыканье _____________
  моя0           → m a j a0
  пое0хать       → p a j e0 h a t'
  бежи0т         → b' i zh y0 t
  бе0лого        → b' e0 l a v a
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' i i
  де0тский       → d' e0 t s k' i j
  жи0ть          → zh y0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s' n' a
  ба0нтик        → b a0 n' t' i k
  ле0стница      → l' e0 s' t' n' i c a
  молоко0        → m a l a k o0
  хорошо0        → h a r a sh o0
  жена0          → zh y n a0
  ше0сть         → sh e0 s' t'
  цена0          → c y n a0
  тру0бка        → t r u0 b k a
  сде0лать       → s' d' e0 l a t'
  гри0б          → g r' i0 b
  моро0з         → m a r o0 z
  но0ж           → n o0 zh
  я0ркого        → j a0 r k a v a
  сши0ть         → s sh y0 t'
  мо0ются        → m o0 j u t s' a
  че0стный       → ch e0 s t n y j
  по0здно        → p o0 z d n a
  со0лнце        → s o0 l n c y



# 7. Регрессивная ассимиляция по звонкости

In [249]:
def rule_8_voicing_assimilation(phonemes: List[Phoneme]) -> List[Phoneme]:
    result = phonemes[:]
    for i in range(len(result) - 2, -1, -1):
        left, right = result[i], result[i + 1]
        if not (left.base in OBSTRUENTS and right.base in OBSTRUENTS):
            continue

        if (left.base == 't' and right.base == 'v'
                and i > 0 and result[i - 1].base == 's'):
            continue

        if right.base in VOICELESS_OBSTRUENTS and left.base in VOICED_TO_VOICELESS:
            result[i] = replace(left, base=VOICED_TO_VOICELESS[left.base])
        elif (right.base in VOICED_OBSTRUENTS
                and left.base in VOICELESS_TO_VOICED
                and right.base != 'v'):  # ← v не вызывает озвончения слева
            result[i] = replace(left, base=VOICELESS_TO_VOICED[left.base])
    return result


state = {w: rule_8_voicing_assimilation(state[w]) for w in TEST_WORDS}
show_step("8. Ассимиляция по звонкости", state)

_____________ 8. Ассимиляция по звонкости _____________
  моя0           → m a j a0
  пое0хать       → p a j e0 h a t'
  бежи0т         → b' i zh y0 t
  бе0лого        → b' e0 l a v a
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' i i
  де0тский       → d' e0 t s k' i j
  жи0ть          → zh y0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s' n' a
  ба0нтик        → b a0 n' t' i k
  ле0стница      → l' e0 s' t' n' i c a
  молоко0        → m a l a k o0
  хорошо0        → h a r a sh o0
  жена0          → zh y n a0
  ше0сть         → sh e0 s' t'
  цена0          → c y n a0
  тру0бка        → t r u0 p k a
  сде0лать       → z' d' e0 l a t'
  гри0б          → g r' i0 b
  моро0з         → m a r o0 z
  но0ж           → n o0 zh
  я0ркого        → j a0 r k a v a
  сши0ть         → s sh y0 t'
  мо0ются        → m o0 j u t s' a
  че0стный       → ch e0 s t n y j
  по0здно        → p o0 z d n a
  со0лнце        → s o0 l n c y



# 8. Конечное оглушение

In [250]:
def rule_9_final_devoicing(phonemes: List[Phoneme]) -> List[Phoneme]:
    if not phonemes:
        return phonemes
    result = phonemes[:]
    last = result[-1]
    if last.base in VOICED_OBSTRUENTS:
        result[-1] = replace(last, base=VOICED_TO_VOICELESS[last.base])
    return result


state = {w: rule_9_final_devoicing(state[w]) for w in TEST_WORDS}
show_step("9. Конечное оглушение", state)

_____________ 9. Конечное оглушение _____________
  моя0           → m a j a0
  пое0хать       → p a j e0 h a t'
  бежи0т         → b' i zh y0 t
  бе0лого        → b' e0 l a v a
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' i i
  де0тский       → d' e0 t s k' i j
  жи0ть          → zh y0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s' n' a
  ба0нтик        → b a0 n' t' i k
  ле0стница      → l' e0 s' t' n' i c a
  молоко0        → m a l a k o0
  хорошо0        → h a r a sh o0
  жена0          → zh y n a0
  ше0сть         → sh e0 s' t'
  цена0          → c y n a0
  тру0бка        → t r u0 p k a
  сде0лать       → z' d' e0 l a t'
  гри0б          → g r' i0 p
  моро0з         → m a r o0 s
  но0ж           → n o0 sh
  я0ркого        → j a0 r k a v a
  сши0ть         → s sh y0 t'
  мо0ются        → m o0 j u t s' a
  че0стный       → ch e0 s t n y j
  по0здно        → p o0 z d n a
  со0лнце        → s o0 l n c y



# 9. Слияние согласных кластеров

In [251]:
CLUSTER_MAP: Dict[tuple, str] = {
    ("s",  "sh"): "sh",
    ("z",  "sh"): "sh",
    ("s",  "zh"): "zh",
    ("z",  "zh"): "zh",
    ("t",  "ch"): "ch",
    ("d",  "ch"): "ch",
    ("t",  "s"):  "c",
    ("d",  "s"):  "c",
}


def rule_11_cluster_assimilation(phonemes: List[Phoneme]) -> List[Phoneme]:
    result: List[Phoneme] = []
    i = 0
    while i < len(phonemes):
        if i + 1 < len(phonemes):
            left, right = phonemes[i], phonemes[i + 1]
            key = (left.base, right.base)
            if key in CLUSTER_MAP:
                merged_base = CLUSTER_MAP[key]
                merged = Phoneme(
                    base=merged_base,
                    palatalized=right.palatalized and merged_base not in ALWAYS_HARD,
                )
                result.append(merged)
                i += 2
                continue
        result.append(phonemes[i])
        i += 1
    return result


state = {w: rule_11_cluster_assimilation(state[w]) for w in TEST_WORDS}
show_step("11. Слияние кластеров", state)

_____________ 11. Слияние кластеров _____________
  моя0           → m a j a0
  пое0хать       → p a j e0 h a t'
  бежи0т         → b' i zh y0 t
  бе0лого        → b' e0 l a v a
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' i i
  де0тский       → d' e0 c k' i j
  жи0ть          → zh y0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s' n' a
  ба0нтик        → b a0 n' t' i k
  ле0стница      → l' e0 s' t' n' i c a
  молоко0        → m a l a k o0
  хорошо0        → h a r a sh o0
  жена0          → zh y n a0
  ше0сть         → sh e0 s' t'
  цена0          → c y n a0
  тру0бка        → t r u0 p k a
  сде0лать       → z' d' e0 l a t'
  гри0б          → g r' i0 p
  моро0з         → m a r o0 s
  но0ж           → n o0 sh
  я0ркого        → j a0 r k a v a
  сши0ть         → sh y0 t'
  мо0ются        → m o0 j u c a
  че0стный       → ch e0 s t n y j
  по0здно        → p o0 z d n a
  со0лнце        → s o0 l n c y



# 10. Непроизносимые согласные

In [252]:
# ключ — кортеж base'ов, значение — индексы позиций, которые ВЫПАДАЮТ
SILENT_DROP_PATTERNS: Dict[tuple, tuple] = {
    ("s", "t", "n"):           (1,),
    ("z", "d", "n"):           (1,),
    ("s", "t", "l"):           (1,),
    ("s", "t", "s", "k"):      (1,),
    ("n", "t", "s", "k"):      (1,),
    ("n", "d", "s", "k"):      (1,),
    ("n", "g", "s", "k"):      (1,),
    ("r", "g", "s", "k"):      (1,),
    ("n", "t", "s", "t", "v"): (1,),
    ("f", "s", "t", "v"):      (0,),
}


def rule_12_silent_consonants(phonemes: List[Phoneme]) -> List[Phoneme]:
    patterns_sorted = sorted(SILENT_DROP_PATTERNS.items(), key=lambda kv: -len(kv[0]))

    result: List[Phoneme] = []
    i = 0
    while i < len(phonemes):
        matched = False
        for pattern, drop_indices in patterns_sorted:
            n = len(pattern)
            if i + n <= len(phonemes):
                window_bases = tuple(p.base for p in phonemes[i:i + n])
                if window_bases == pattern:
                    for j, ph in enumerate(phonemes[i:i + n]):
                        if j not in drop_indices:
                            result.append(ph)
                    i += n
                    matched = True
                    break
        if not matched:
            result.append(phonemes[i])
            i += 1
    return result


state = {w: rule_12_silent_consonants(state[w]) for w in TEST_WORDS}
show_step("12. Непроизносимые согласные", state)

_____________ 12. Непроизносимые согласные _____________
  моя0           → m a j a0
  пое0хать       → p a j e0 h a t'
  бежи0т         → b' i zh y0 t
  бе0лого        → b' e0 l a v a
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' i i
  де0тский       → d' e0 c k' i j
  жи0ть          → zh y0 t'
  щу0ка          → sc u0 k a
  пе0сня         → p' e0 s' n' a
  ба0нтик        → b a0 n' t' i k
  ле0стница      → l' e0 s' n' i c a
  молоко0        → m a l a k o0
  хорошо0        → h a r a sh o0
  жена0          → zh y n a0
  ше0сть         → sh e0 s' t'
  цена0          → c y n a0
  тру0бка        → t r u0 p k a
  сде0лать       → z' d' e0 l a t'
  гри0б          → g r' i0 p
  моро0з         → m a r o0 s
  но0ж           → n o0 sh
  я0ркого        → j a0 r k a v a
  сши0ть         → sh y0 t'
  мо0ются        → m o0 j u c a
  че0стный       → ch e0 s n y j
  по0здно        → p o0 z n a
  со0лнце        → s o0 l n c y



# 11. Упрощение геминат

In [253]:
def rule_13_soften_geminate(phonemes: List[Phoneme], morph_boundaries: Set[int] | None = None) -> List[Phoneme]:
    if not phonemes:
        return phonemes

    result: List[Phoneme] = [phonemes[0]]
    for p in phonemes[1:]:
        prev = result[-1]
        if not (is_consonant(p) and is_consonant(prev) and p.base == prev.base):
            result.append(p)
            continue

        if not prev.palatalized and p.palatalized:
            result[-1] = p
            continue

        if p.palatalized == prev.palatalized:
            if morph_boundaries is not None and p.src_pos in morph_boundaries:
                result.append(p)
            continue

        result.append(p)
    return result

# Пайплайн

In [254]:
def transcribe(stressed_word: str, morph_str: str | None = None, use_morphemes: bool = True) -> str:
    boundaries = (
        parse_morpheme_boundaries(morph_str)
        if use_morphemes and morph_str is not None
        else None
    )

    phonemes = to_phonemes(stressed_word, boundaries, use_morphemes)  # уже включает правила 1, 3, 10
    phonemes = rule_2_intervocalic_jot(phonemes)
    phonemes = rule_4_softness_assimilation(phonemes)
    phonemes = rule_5_akanye(phonemes)
    phonemes = rule_6_ikanye(phonemes)
    phonemes = rule_7_ykanye(phonemes)
    phonemes = rule_8_voicing_assimilation(phonemes)
    phonemes = rule_9_final_devoicing(phonemes)
    phonemes = rule_11_cluster_assimilation(phonemes)
    phonemes = rule_12_silent_consonants(phonemes)
    phonemes = rule_13_soften_geminate(phonemes, boundaries) if use_morphemes else phonemes
    return phonemes_to_str(phonemes)


# проверка на тестовых данных
for w in TEST_WORDS[:9]:
    print(f"  {w:<14} → {transcribe(w)}")

  моя0           → m a j a0
  пое0хать       → p a j e0 h a t'
  бежи0т         → b' i zh y0 t
  бе0лого        → b' e0 l a v a
  краси0вая      → k r a s' i0 v a i
  си0няя         → s' i0 n' i i
  де0тский       → d' e0 c k' i j
  жи0ть          → zh y0 t'
  щу0ка          → sc u0 k a


# Основная логика

## Нормализация орфоэпической транскрипции

In [255]:
import re

SERVICE_TOKENS: Set[str] = {'*', '+', '/', '~', '=', '^', '-'}

def normalize_reference(ref: str) -> str:
    tokens: list[str] = []
    for tok in ref.split():
        if tok in SERVICE_TOKENS:
            continue
        tok = re.sub(r'[1-4]', '', tok)
        tokens.append(tok)
    return ' '.join(tokens)


print("Эталон → нормализованный:")
for ref in df[COL_ORTHOEPIC_TRANS].head(10):
    print(f"  {ref}")
    print(f"  → {normalize_reference(ref)}")
    print()

Эталон → нормализованный:
  * a1 p s a1 l' u0 t + n + a4
  → a p s a l' u0 t n a

  * a1 p h a0 z' + i4 c
  → a p h a0 z' i c

  * a0 v g u4 s t
  → a0 v g u s t

  * a0 v g u4 s' t' / i4
  → a0 v g u s' t' i

  * a1 f t a2 * b' i1 a1 * g r a0 f' i4 / i4
  → a f t a b' i a g r a0 f' i i

  * a1 f t o0 b u4 s' / i4
  → a f t o0 b u s' i

  * a1 f t o0 b u4 s + n / a4 j
  → a f t o0 b u s n a j

  * a1 f t o0 b u4 s / a4 m
  → a f t o0 b u s a m

  * a1 f t a1 m a0 t
  → a f t a m a0 t

  * a1 f t a1 m a0 t / a4
  → a f t a m a0 t a



# Результаты

## С учетом и без учета морфем

In [256]:
COL_PRED_MORPH = 'Предсказание (с морфемами)'
COL_PRED_NO_MORPH = 'Предсказание (без морфем)'
COL_REF_NORM = 'Эталон (нормализованный)'

df[COL_PRED_MORPH] = df.apply(
    lambda r: transcribe(r[COL_STRESSED_WORD], r[COL_MORPHEMES], use_morphemes=True),
    axis=1,
)
df[COL_PRED_NO_MORPH] = df[COL_STRESSED_WORD].map(
    lambda w: transcribe(w, None, use_morphemes=False)
)
df[COL_REF_NORM] = df[COL_ORTHOEPIC_TRANS].map(normalize_reference)

df['match_morph'] = df[COL_PRED_MORPH] == df[COL_REF_NORM]
df['match_no_morph'] = df[COL_PRED_NO_MORPH] == df[COL_REF_NORM]

total = len(df)
print(f"С морфемами:  {df['match_morph'].sum()} / {total} ({df['match_morph'].sum() / total * 100:.2f}%)")
print(f"Без морфем:   {df['match_no_morph'].sum()} / {total} ({df['match_no_morph'].sum() / total * 100:.2f}%)")
print(f"Дельта:       {df['match_morph'].sum() - df['match_no_morph'].sum():+d}")

mismatches = df[~df['match_morph']][[COL_WORD, COL_STRESSED_WORD, COL_PRED_MORPH, COL_REF_NORM]]
print(f"Несовпадений (с морфемами): {len(mismatches)}")
mismatches.head(10)


С морфемами:  6077 / 7218 (84.19%)
Без морфем:   5909 / 7218 (81.86%)
Дельта:       +168
Несовпадений (с морфемами): 1141


,Слово,Слово с ударением,Предсказание (с морфемами),Эталон (нормализованный)
11,автоматная,автома0тная,a f t a m a0 t n a i,a f t a m a0 t n a j a
24,аккуратная,аккура0тная,a k u r a0 t n a i,a k u r a0 t n a j a
29,алексеевич,алексе0евич,a l' i k s' e0 i v' i ch,a l' i k s' e0 j i v' i ch
30,алексеевича,алексе0евича,a l' i k s' e0 i v' i ch a,a l' i k s' e0 j i v' i ch a
31,алексеевичем,алексе0евичем,a l' i k s' e0 i v' i ch i m,a l' i k s' e0 j i v' i ch i m
32,алексеевна,алексе0евна,a l' i k s' e0 i v n a,a l' i k s' e0 j i v n a
33,алексеевной,алексе0евной,a l' i k s' e0 i v n a j,a l' i k s' e0 j i v n a j
36,алексея,алексе0я,a l' i k s' e0 i,a l' i k s' e0 j a
39,амбиция,амби0ция,a m b' i0 c y i,a m b' i0 c y j a
62,архитектурная,архитекту0рная,a r h' i t' i k t u0 r n a i,a r h' i t' i k t u0 r n a j a


In [257]:
mismatches_no_morph = df[~df['match_no_morph']][[COL_WORD, COL_STRESSED_WORD, COL_PRED_NO_MORPH, COL_REF_NORM]]
print(f"Несовпадений без морфем: {len(mismatches_no_morph)}")
mismatches_no_morph.head(10)

Несовпадений без морфем: 1309


,Слово,Слово с ударением,Предсказание (без морфем),Эталон (нормализованный)
11,автоматная,автома0тная,a f t a m a0 t n a i,a f t a m a0 t n a j a
21,адресованном,адресо0ванном,a d r' i s o0 v a n n a m,a d r' i s o0 v a n a m
24,аккуратная,аккура0тная,a k k u r a0 t n a i,a k u r a0 t n a j a
25,аккуратно,аккура0тно,a k k u r a0 t n a,a k u r a0 t n a
26,аккуратный,аккура0тный,a k k u r a0 t n y j,a k u r a0 t n y j
29,алексеевич,алексе0евич,a l' i k s' e0 i v' i ch,a l' i k s' e0 j i v' i ch
30,алексеевича,алексе0евича,a l' i k s' e0 i v' i ch a,a l' i k s' e0 j i v' i ch a
31,алексеевичем,алексе0евичем,a l' i k s' e0 i v' i ch i m,a l' i k s' e0 j i v' i ch i m
32,алексеевна,алексе0евна,a l' i k s' e0 i v n a,a l' i k s' e0 j i v n a
33,алексеевной,алексе0евной,a l' i k s' e0 i v n a j,a l' i k s' e0 j i v n a j


In [258]:
helped = df[df['match_morph'] & ~df['match_no_morph']][[COL_WORD, COL_PRED_NO_MORPH, COL_PRED_MORPH, COL_REF_NORM]]
hurt = df[~df['match_morph'] & df['match_no_morph']][[COL_WORD, COL_PRED_NO_MORPH, COL_PRED_MORPH, COL_REF_NORM]]
print(f"Морфемы помогли: {len(helped)} слов")
print(f"Морфемы испортили: {len(hurt)} слов")

Морфемы помогли: 231 слов
Морфемы испортили: 63 слов
